# Quantum Error Correction with the 3-Qubit Repetition Code

In this example we will use a 3 qubit repetition code to detect and correct single Pauli X bit flip errors using pairwise parity checks. See https://en.wikipedia.org/wiki/Repetition_code for more info.

In [ ]:
from guppylang import guppy
from guppylang.std.array import array
from guppylang.std.qsystem.random import RNG
from guppylang.std.qsystem.utils import get_current_shot
from guppylang.std.quantum import cx, discard_array, h, measure_array, qubit, x

We will first create our data and ancilla qubits and then create some quantum information to be protected and encode information from q[0] by repetition, i.e. |0> -> |000>, |1> -> |111>

In [ ]:
# Data / physical qubits
q = array(qubit() for _ in range(3))
# Ancilla qubits used for parity checks
anc = array(qubit() for _ in range(2))

# Create some quantum information to be protected
h(q[0])

# Encode information from q[0] by repetition, i.e. |0> -> |000>, |1> -> |111>
cx(q[0], q[1])
cx(q[0], q[2])

We will then seed guppy's random number generator using the current shot number and artificially induce a single x bit flip error in our program.

In [ ]:
@guppy
def flip_random_bit(q: array[qubit, 3], rng: RNG) -> None:
    # Each qubit has a 1/4 chance to be hit by the flip, 1/4 chance nothing happens
    r = rng.random_int_bounded(4)
    if r < 3:
        x(q[r])
        output("Flipped qubit", r)

# Randomly induce a bit-flip on one qubit to simulate an error.
rng = RNG(get_current_shot())
flip_random_bit(q, rng)

We then extract our syndrome from the ancilla qubits and use the 3 qubit repetition code lookup table to check for disagreements between qubits.

In [ ]:
# Perform minimum required parity checks between q[0]q[1] pair and q[1]q[2] pair
for i in range(2):
    cx(q[i], anc[i])
    cx(q[i+1], anc[i])

s = measure_array(anc)
# q[0]q[1] disagree, q[1]q[2] agree -> q[0] is bad
if s[0] and not s[1]:
    correct_qubit = 0
    
# q[0]q[1] disagree, q[1]q[2] disagree -> q[1] is bad
elif s[0] and s[1]:
    correct_qubit = 1

# q[0]q[1] agree, q[1]q[2] disagree -> q[2] is bad
elif not s[0] and s[1]:
    correct_qubit = 2

# q[0]q[1] agree, q[1]q[2] agree, no corrections necessary
else:
    correct_qubit = 3

if correct_qubit < 3:
    x(q[correct_qubit])
    output("Corrected qubit", correct_qubit)

discard_array(q)
rng.discard()